# Global linear SRM: feature-recipe comparison

This notebook evaluates one global linear SRM model with two **preselected, fold-specific** feature recipes: the original FRDA-only selector and the supervisor-requested control-aware selector. It does not select imaging features.

**Validation contract.** Participant defines every outer split. Feature selection is already restricted to each outer-training fold. SRM tuning, scaling, and coefficients use outer-training FRDA only. The frozen pipeline then scores held-out FRDA and held-out controls.


## 1. Data, folds, and selected-feature recipes

The two recipes use the same 70-feature candidate panel and the same persisted participant folds. Controls never contribute to SRM scaling or coefficients.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.trackfa_pairs import trackfa_pairs_to_long
from src.eval.control_aware_selection import wide_cohort_to_pair_long
from src.features.panels import a_priori_70_feature_names
from src.reporting.experiment_artifacts import (
    read_experiment_contract,
    read_table_artifact,
    write_table_artifact,
)

RUN_ID = "trackfa_70_feature_comparison_v1"
RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
SELECTION_DIR = RUN_DIR / "selections"
manifest, folds = read_experiment_contract(RUN_DIR)
feature_names = a_priori_70_feature_names()
frda_recipe = read_table_artifact(
    SELECTION_DIR / "frda_only_features_by_fold.csv",
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
)
control_recipe = read_table_artifact(
    SELECTION_DIR / "control_aware_features_by_fold.csv",
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
)
feature_recipes = pd.concat([frda_recipe, control_recipe], ignore_index=True)

pairs = pd.read_csv(manifest["data_path"])
frda_long = trackfa_pairs_to_long(pairs)
wide_path = Path(manifest["data_path"]).with_name("trackfa_merged_wide.csv")
wide = pd.read_csv(wide_path, low_memory=False)
control_long = wide_cohort_to_pair_long(wide, feature_names, cohort_value=1)
control_meta = wide.loc[pd.to_numeric(wide["study_group"], errors="coerce").eq(1), ["ID", "age"]].copy()
control_meta["subject"] = control_meta["ID"].astype(str).str.replace(r"^TRACKFA_", "", regex=True)
control_meta["age"] = pd.to_numeric(control_meta["age"], errors="coerce")
control_long = control_long.merge(control_meta[["subject", "age"]].drop_duplicates("subject"), on="subject", how="left")

print(f"Run: {RUN_ID}")
display(pd.DataFrame([
    {"Cohort": "FRDA", "Participants": frda_long["subject"].nunique(), "Annual pairs": frda_long["pair_id"].nunique()},
    {"Cohort": "Control", "Participants": control_long["subject"].nunique(), "Annual pairs": control_long["pair_id"].nunique()},
]))
display(pd.DataFrame([
    {"Selection recipe": "FRDA-only", "Features per fold": int(frda_recipe.groupby("outer_fold")["selected"].sum().mode().iloc[0])},
    {"Selection recipe": "Control-aware", "Features per fold": int(control_recipe.groupby("outer_fold")["selected"].sum().mode().iloc[0])},
]))

from src.eval.recipe_models import run_srm_recipe_comparison

GUARDRAILS = pd.DataFrame([
    {"Step": "Outer split", "Training information": "Persisted participant folds", "Held-out information excluded": "FRDA and control test participants"},
    {"Step": "Feature selection", "Training information": "Fold-specific recipe", "Held-out information excluded": "All outer-test measurements"},
    {"Step": "Scaling", "Training information": "Outer-training FRDA visits", "Held-out information excluded": "FRDA test and all control values"},
    {"Step": "SRM fit", "Training information": "Outer-training FRDA pair deltas", "Held-out information excluded": "FRDA test and controls"},
])
display(GUARDRAILS)


Run: trackfa_70_feature_comparison_v1


,Cohort,Participants,Annual pairs
0,FRDA,117,207
1,Control,95,190


,Selection recipe,Features per fold
0,FRDA-only,16
1,Control-aware,16


,Step,Training information,Held-out information excluded
0,Outer split,Persisted participant folds,FRDA and control test participants
1,Feature selection,Fold-specific recipe,All outer-test measurements
2,Scaling,Outer-training FRDA visits,FRDA test and all control values
3,SRM fit,Outer-training FRDA pair deltas,FRDA test and controls


## 2. Out-of-fold evaluation

SRM regularisation is chosen by grouped inner CV inside each outer-training FRDA fold. The primary result is pooled annual paired Cohen's \(d_z\); interval-specific results show whether progression is consistent across V1-to-V2 and V2-to-V3.


In [2]:
result = run_srm_recipe_comparison(
    frda_long,
    control_long,
    folds,
    feature_recipes,
    run_id=RUN_ID,
    inner_folds=3,
    seed=int(manifest["seed"]),
    n_boot=500,
)

MODEL_DIR = RUN_DIR / "models" / "srm_global_linear"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
write_table_artifact(MODEL_DIR / "oof_visit_scores.csv", result["oof_visit_scores"], schema="oof_visit_scores", manifest=manifest)
write_table_artifact(MODEL_DIR / "performance.csv", result["performance"], schema="performance", manifest=manifest)
write_table_artifact(MODEL_DIR / "coefficients.csv", result["coefficients"], schema="coefficients", manifest=manifest)
result["fold_parameters"].to_csv(MODEL_DIR / "fold_parameters.csv", index=False)
result["tuning"].to_csv(MODEL_DIR / "inner_tuning.csv", index=False)
result["site_diagnostics"].to_csv(MODEL_DIR / "site_diagnostics.csv", index=False)
result["comparison"].to_csv(MODEL_DIR / "headline_comparison.csv", index=False)

headline_columns = [
    "selection_strategy",
    "frda_pooled_annual_d_z", "frda_pooled_annual_ci_low", "frda_pooled_annual_ci_high",
    "frda_pooled_annual_n_participants", "frda_pooled_annual_n_pairs",
    "control_pooled_annual_d_z", "control_pooled_annual_ci_low", "control_pooled_annual_ci_high",
    "control_pooled_annual_n_participants", "control_pooled_annual_n_pairs",
    "signed_frda_control_contrast", "absolute_control_d_z",
    "frda_v1_v2_d_z", "frda_v2_v3_d_z", "control_v1_v2_d_z", "control_v2_v3_d_z",
    "frda_interval_gap",
]
headline = result["comparison"][[column for column in headline_columns if column in result["comparison"]]].copy()
headline = headline.rename(columns={"selection_strategy": "Feature selection"})
print("Global SRM held-out comparison")
display(headline.round(3))


Global SRM held-out comparison


,Feature selection,frda_pooled_annual_d_z,frda_pooled_annual_ci_low,frda_pooled_annual_ci_high,frda_pooled_annual_n_participants,frda_pooled_annual_n_pairs,control_pooled_annual_d_z,control_pooled_annual_ci_low,control_pooled_annual_ci_high,control_pooled_annual_n_participants,control_pooled_annual_n_pairs,signed_frda_control_contrast,absolute_control_d_z,frda_v1_v2_d_z,frda_v2_v3_d_z,control_v1_v2_d_z,control_v2_v3_d_z,frda_interval_gap
0,control_aware,0.749,0.614,0.900,117,207,-0.011,-0.168,0.168,67,126,0.760,0.011,0.944,0.579,-0.019,-0.003,0.365
1,frda_only,0.749,0.610,0.912,117,207,0.033,-0.121,0.222,67,126,0.715,0.033,0.959,0.569,0.007,0.059,0.390


## 3. Fold variability and site diagnostic

The tables below are diagnostics, not selection criteria. Site association is tested on held-out score changes; it does not alter the fitted model in this run.


In [3]:
fold_effects = []
for keys, part in result["oof_visit_scores"].groupby(["selection_strategy", "cohort", "outer_fold"]):
    strategy, cohort, fold = keys
    paired = part.pivot_table(index="pair_id", columns="visit", values="score", aggfunc="mean")
    delta = paired[2] - paired[1]
    fold_effects.append({
        "Feature selection": strategy,
        "Cohort": cohort,
        "Fold": int(fold),
        "Pairs": int(delta.notna().sum()),
        "d_z": float(delta.mean() / delta.std(ddof=1)) if delta.notna().sum() >= 2 and delta.std(ddof=1) else np.nan,
    })
display(pd.DataFrame(fold_effects).round(3))
site_display = result["site_diagnostics"].rename(columns={
    "selection_strategy": "Feature selection", "cohort": "Cohort",
    "site_r2_delta": "Site partial R2", "site_p_value": "Site p-value",
})
display(site_display[[c for c in ["Feature selection", "Cohort", "n", "site_levels", "Site partial R2", "Site p-value"] if c in site_display]].round(3))


,Feature selection,Cohort,Fold,Pairs,d_z
0,control_aware,Control,1,28,0.129
1,control_aware,Control,2,21,0.043
2,control_aware,Control,3,24,-0.169
3,control_aware,Control,4,27,0.085
4,control_aware,Control,5,26,-0.199
5,control_aware,FRDA,1,42,0.542
6,control_aware,FRDA,2,44,0.979
7,control_aware,FRDA,3,40,0.833
8,control_aware,FRDA,4,41,1.023
9,control_aware,FRDA,5,40,0.592


,Feature selection,Cohort,n,site_levels,Site partial R2,Site p-value
0,frda_only,FRDA,207,6,0.075,0.007
1,frda_only,Control,126,6,0.052,0.258
2,control_aware,FRDA,207,6,0.077,0.006
3,control_aware,Control,126,6,0.047,0.315


## 4. Model parameters and interpretation

Coefficients are conditional weights on standardised MRI variables. Their signs must not be interpreted as each feature's marginal longitudinal direction. Detailed marginal-versus-conditional interpretation belongs in the locked-winner feature-importance notebook.


In [4]:
parameter_display = result["fold_parameters"][[
    "selection_strategy", "outer_fold", "feature_count", "ridge", "covariance_shrinkage", "z_clip",
    "train_frda_participants", "train_frda_pairs", "test_frda_participants", "test_control_participants",
]].rename(columns={
    "selection_strategy": "Feature selection", "outer_fold": "Fold", "feature_count": "Features",
    "train_frda_participants": "Train FRDA N", "train_frda_pairs": "Train FRDA pairs",
    "test_frda_participants": "Test FRDA N", "test_control_participants": "Test control N",
})
display(parameter_display)
print("Machine-readable artifacts:", MODEL_DIR)


,Feature selection,Fold,Features,ridge,covariance_shrinkage,z_clip,Train FRDA N,Train FRDA pairs,Test FRDA N,Test control N
0,frda_only,1,16,0.0,0.45,NaN,93,165,24,14
1,frda_only,2,16,0.0,0.45,2.75,93,163,24,13
2,frda_only,3,16,0.0,0.45,NaN,94,167,23,13
3,frda_only,4,16,0.0,0.45,NaN,94,166,23,14
4,frda_only,5,16,0.0,0.35,NaN,94,167,23,13
5,control_aware,1,16,0.0,0.45,NaN,93,165,24,14
6,control_aware,2,16,0.0,0.45,NaN,93,163,24,13
7,control_aware,3,16,0.0,0.45,NaN,94,167,23,13
8,control_aware,4,16,0.0,0.45,NaN,94,166,23,14
9,control_aware,5,16,0.0,0.35,NaN,94,167,23,13


Machine-readable artifacts: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/models/srm_global_linear


## Interpretation boundary

All headline effects above are out-of-fold estimates. A later locked-winner interpretation step may refit one model on all FRDA data to produce deployment means, standard deviations, clipping settings, and coefficients. That full-data fit must not replace these held-out performance estimates.
